In [1]:
import pandas as pd

training_start_date = '2023-01-01'

# Load data
train_df = pd.read_csv('~/data/Germany_time_zone/all/train_with_all.csv')
test_df = pd.read_csv('~/data/Germany_time_zone/all/test_with_all.csv')

# Filter train_df to include only data from 2020 onward
train_df = train_df[pd.to_datetime(train_df['ds']) >= pd.Timestamp(training_start_date)]

# Convert to UTC while handling DST ambiguities
train_df['ds'] = pd.to_datetime(train_df['ds']).dt.tz_localize('Europe/Berlin', ambiguous='infer').dt.tz_convert('UTC')
test_df['ds'] = pd.to_datetime(test_df['ds']).dt.tz_localize('Europe/Berlin', ambiguous='infer').dt.tz_convert('UTC')

# Check date ranges
print("First date in train_df:", train_df['ds'].min())
print("Last date in train_df:", train_df['ds'].max())
print("First date in test_df:", test_df['ds'].min())
print("Last date in test_df:", test_df['ds'].max())


calendar_features = ['month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 
                     'hour_sin', 'hour_cos', 'is_holiday', 'is_weekend']
gas_price_features = ['gas_price']
load_features = ['load', 'co2_emission_allowances', 'synthetic_price']
renewable_features = ['renewable', 'non_renewable']
cross_border_features = ['cross_border_trading']

# Define the exogenous variable lists
stat_exog_list = [] 
hist_exog_list = (
    calendar_features
    + gas_price_features
    + ['synthetic_price', 'co2_emission_allowances']
    # + load_features
    # + renewable_features
    # + cross_border_features
)

futr_exog_list = (
    calendar_features
)



First date in train_df: 2022-12-31 23:00:00+00:00
Last date in train_df: 2023-12-31 22:00:00+00:00
First date in test_df: 2023-12-31 23:00:00+00:00
Last date in test_df: 2024-12-31 22:00:00+00:00


In [2]:
train_df.head()

,ds,y,unique_id,day_of_week,month,hour,is_weekend,is_holiday,month_sin,month_cos,...,day_of_week_cos,hour_sin,hour_cos,gas_price,cross_border_trading,non_renewable,renewable,load,co2_emission_allowances,synthetic_price
37273,2022-12-31 23:00:00+00:00,-5.17,electricity_prices,6,1,0,1,1,0.5,0.866025,...,0.62349,0.000000,1.000000,78.5235,-12557.67,8972.50,36606.37,38346.04,83.41,176.134
37274,2023-01-01 00:00:00+00:00,-1.07,electricity_prices,6,1,1,1,1,0.5,0.866025,...,0.62349,0.258819,0.965926,78.5235,-15382.57,8766.41,37638.88,37777.20,83.41,176.134
37275,2023-01-01 01:00:00+00:00,-1.47,electricity_prices,6,1,2,1,1,0.5,0.866025,...,0.62349,0.500000,0.866025,78.5235,-16357.49,8743.59,37838.87,36939.93,83.41,176.134
37276,2023-01-01 02:00:00+00:00,-5.08,electricity_prices,6,1,3,1,1,0.5,0.866025,...,0.62349,0.707107,0.707107,78.5235,-14005.91,8742.11,35322.31,35932.39,83.41,176.134
37277,2023-01-01 03:00:00+00:00,-4.49,electricity_prices,6,1,4,1,1,0.5,0.866025,...,0.62349,0.866025,0.500000,78.5235,-14543.99,8704.77,34805.70,35486.24,83.41,176.134


In [3]:
test_df.head()

,ds,y,unique_id,day_of_week,month,hour,is_weekend,is_holiday,month_sin,month_cos,...,day_of_week_cos,hour_sin,hour_cos,gas_price,cross_border_trading,non_renewable,renewable,load,co2_emission_allowances,synthetic_price
0,2023-12-31 23:00:00+00:00,0.10,electricity_prices,0,1,0,0,1,0.5,0.866025,...,1.0,0.000000,1.000000,31.574,-11728.9125,8378.536596,41778.25343,40170.0900,66.6,84.047273
1,2024-01-01 00:00:00+00:00,0.01,electricity_prices,0,1,1,0,1,0.5,0.866025,...,1.0,0.258819,0.965926,31.574,-11129.9725,8284.442818,41522.96191,38818.1500,66.6,84.047273
2,2024-01-01 01:00:00+00:00,0.00,electricity_prices,0,1,2,0,1,0.5,0.866025,...,1.0,0.500000,0.866025,31.574,-11635.4075,8282.031575,41346.84615,37847.5650,66.6,84.047273
3,2024-01-01 02:00:00+00:00,-0.01,electricity_prices,0,1,3,0,1,0.5,0.866025,...,1.0,0.707107,0.707107,31.574,-11492.7000,8318.301450,40363.41746,37123.2325,66.6,84.047273
4,2024-01-01 03:00:00+00:00,-0.03,electricity_prices,0,1,4,0,1,0.5,0.866025,...,1.0,0.866025,0.500000,31.574,-10593.8850,8309.993639,40234.70911,36753.5150,66.6,84.047273


Train the Model

In [4]:
print(        
    stat_exog_list
      )
print(
    hist_exog_list,
    )
print(    
    futr_exog_list,
     )

[]
['month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'hour_sin', 'hour_cos', 'is_holiday', 'is_weekend', 'gas_price', 'synthetic_price', 'co2_emission_allowances']
['month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'hour_sin', 'hour_cos', 'is_holiday', 'is_weekend']


In [5]:
from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoNBEATSx
import optuna

horizon = 24
historical = 24 * 7
validation_size = int(0.1 * len(train_df))
# validation_size = 24 * 30 * 3 # 3 months 

number_of_samples = 35
model_name = 'AutoNBEATSx'


def config(trial):
    h = horizon
    return {
        "stat_exog_list": trial.suggest_categorical("stat_exog_list", [stat_exog_list]),
        "hist_exog_list": trial.suggest_categorical("hist_exog_list", [hist_exog_list]),
        "futr_exog_list": trial.suggest_categorical("futr_exog_list", [futr_exog_list]),
        "input_size": trial.suggest_categorical(
            "input_size",
            # [x * 24 for x in [1, 3, 4, 7, 10, 14]]
            [historical]
        ),
        "max_steps": trial.suggest_int("max_steps", 500, 10000, step=500),  # Maximum steps (quantized by 1000)
        "step_size": trial.suggest_categorical("step_size",
                                               [1, h]
                                               # [h]
                                               ),        
        "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64, 128, 256, 512]),  # Batch size
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),
        "random_seed": trial.suggest_int("random_seed", 1, 20), 
        "scaler_type": trial.suggest_categorical("scaler_type", [None, "robust", "standard"]),  # Scaler type
        "windows_batch_size": trial.suggest_categorical("windows_batch_size", [128, 256, 512, 1024, 2048, 4096]),  # Windows batch size
    
    
        # Additional NBEATSx-specific hyperparameters
        "n_harmonics": trial.suggest_int("n_harmonics", 1, 4),            # Number of harmonic oscillations
        "n_polynomials": trial.suggest_int("n_polynomials", 1, 4),          # Degree of polynomial trend basis
        "stack_types": trial.suggest_categorical("stack_types", [['identity', 'trend', 'seasonality']]),
        "n_blocks": trial.suggest_categorical("n_blocks", [[1, 1, 1], [2, 2, 2]]),
        "mlp_units": trial.suggest_categorical(
            "mlp_units",
            [
                3 * [[512, 512]],
                3 * [[1024, 1024]],
                3 * [[2048, 2048]],
                3 * [[256, 256]],
            ]
        ),
        "activation": trial.suggest_categorical("activation", ['ReLU', 'Softplus', 'Tanh', 'SELU', 'LeakyReLU', 'PReLU', 'Sigmoid']),
    }
    
model = AutoNBEATSx(h=horizon,
                  config=config,
                  search_alg=optuna.samplers.TPESampler(),
                  backend='optuna',
                  num_samples=number_of_samples,
                 )


# Train the model on the entire training data just once
nf = NeuralForecast(models=[model], freq='h')
nf.cross_validation(df=train_df, n_windows=3)

[I 2025-02-17 14:45:32,711] A new study created in memory with name: no-name-2c50196a-9809-433d-a200-70ef5257982e
/pfs/data5/home/kit/iai/fn8011/ml/lib64/python3.9/site-packages/optuna/distributions.py:524: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [] which is of type list.
  warnings.warn(message)
/pfs/data5/home/kit/iai/fn8011/ml/lib64/python3.9/site-packages/optuna/distributions.py:524: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains ['month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'hour_sin', 'hour_cos', 'is_holiday', 'is_weekend', 'gas_price', 'synthetic_price', 'co2_emission_allowances'] which is of type list.
  warnings.warn(message)
/pfs/data5/home/kit/iai/fn8011/ml/lib64/python3.9/site-packages/optuna/distributions.py:524: UserWarning: Choices for a categorical distribution 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...
[W 2025-02-17 14:45:49,069] Trial 0 failed with parameters: {'stat_exog_list': [], 'hist_exog_list': ['month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'hour_sin', 'hour_cos', 'is_holiday', 'is_weekend', 'gas_price', 'synthetic_price', 'co2_emission_allowances'], 'futr_exog_list': ['month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'hour_sin', 'hour_cos', 'is_holiday', 'is_weekend'], 'input_size': 168, 'max_steps': 1000, 'step_size': 1, 'batch_size': 512, 'learning_rate': 0.0016957221566410142, 'random_seed': 12, 'scaler_type': None, 'windows_batch_size': 256, 'n_harmonics': 2, 'n_polynomials': 3, 'stack_types': ['identity', 'trend', 'seasonality'], 'n_blocks': [2, 2, 2], 'mlp_units': [[512, 512], [512, 512], [512, 512]], 'activation': 'PReLU'} because of the following error: NameError("name 'exit' is not defined").
Traceback (most recent call last):
  File "/pfs/data5/home/kit/iai/fn8011/ml/lib64/

NameError: name 'exit' is not defined

In [ ]:
results = nf.models[0].results.trials_dataframe()
results.drop(columns='user_attrs_ALL_PARAMS')

In [ ]:
nf.models[0].results.best_trial.user_attrs["ALL_PARAMS"]

In [ ]:
# Optuna
best_config = nf.models[0].results.best_trial.user_attrs["ALL_PARAMS"]

historical = best_config["input_size"]
print(historical)

In [ ]:
import json

# Let's say you have your best params dictionary
best_params = nf.models[0].results.best_trial.user_attrs["ALL_PARAMS"]

# Remove non-serializable keys (e.g., the loss objects)
best_params.pop('loss', None)
best_params.pop('valid_loss', None)

# Save it to a JSON file
with open("../../best/best_config_synthetic_and_co2.json", "w") as f:
    json.dump(best_params, f, indent=4)

In [ ]:
print('done with hyperparameter optimisation')

Test the model

In [ ]:
import pandas as pd

def fill_with_last_week_value(
        forecast_input_data: pd.DataFrame,
        feature: str,
        forecasting_time: pd.Timestamp,
        known_up_to: pd.Timedelta = None
):
    """
    For each row in forecast_input_data, if the row's ds is beyond the feature's 'known_up_to' relative
    to issue_time, we fill the value with the same hour from last week (ds-7d) found in main_df.

    :param forecast_input_data:  The 24h window for day D we are about to forecast.
    :param feature: Which column to fill.
    :param forecasting_time: The date/time we are issuing the forecast.
    :param known_up_to: How far before/after issue_time the feature is known. 
                        If < 0 then before otherwise after.
                        If None => we treat everything after issue_time as unknown.
    :return: A copy of fut_df with the 'feature' column updated.
    """
    # Copy so we don't modify in-place
    forecast_input_data = forecast_input_data.copy()

    if known_up_to is None:
        # Means: if ds > forecasting_time => unknown
        mask_unknown = forecast_input_data['ds'] > forecasting_time
    else:
        # Means: if ds > (forecasting_time + known_up_to) => unknown
        mask_unknown = forecast_input_data['ds'] > (forecasting_time + known_up_to)

    # We'll fill unknown slots from "ds - 7 days" in test_df
    for idx in forecast_input_data[mask_unknown].index:
        target_ds = forecast_input_data.loc[idx, 'ds']
        ds_week_ago = target_ds - pd.Timedelta(days=7)

        # Attempt to find the row in test_df for ds_week_ago
        row_week_ago = test_df.loc[test_df['ds'] == ds_week_ago]

        if not row_week_ago.empty:
            # Fill with that value
            forecast_input_data.at[idx, feature] = row_week_ago[feature].values[0]
        # else:
        #     # If we can't find an exact match, fallback or leave as is
        #     # (could do interpolation or an average)
        #     # Here we fallback to the existing value or fill zero
        #     forecast_input_data.at[idx, feature] = 0

    return forecast_input_data


In [ ]:
import pytz

# Define Germany timezone
germany_tz = pytz.timezone("Europe/Berlin")


historical = best_config["input_size"]
print(historical)

# 2) Prediction Loop
daterange = pd.date_range(
    start=test_df['ds'].min(),
    end=test_df['ds'].max() - pd.Timedelta(hours=historical),
    freq='24H',
)

predictions = []

for d in daterange:
    day_start = d
    day_end = d + pd.Timedelta(hours=historical)

    # 2.1) Gather input data based on historical
    input_data = test_df[(test_df['ds'] >= day_start) & (test_df['ds'] < day_end)].copy()
    if input_data.empty:
        continue

    # 2.2) Define the "forecasting_time" = day_end 11:00 o'clock
    # Calculate issue time in Germany time
    issue_time_germany = day_end.astimezone(germany_tz).replace(hour=11, minute=0, second=0)

    # Convert back to UTC for consistency
    issue_time = issue_time_germany.astimezone(pytz.UTC)

    # 2.3) Fill unknown features
    # Example: load known up to (forecasting_time - 2h),
    #          gas_price known for the entire day,
    #          co2_emissions known only up to forecasting_time, etc.
    input_data = fill_with_last_week_value(input_data, 'load', issue_time,
                                           known_up_to=-pd.Timedelta(hours=2))
    # input_data = fill_with_last_week_value(input_data, 'co2_emissions', issue_time, known_up_to=None)
    # input_data = fill_with_last_week_value(input_data, 'synthetic_price', issue_time, 
    #                                        known_up_to=-pd.Timedelta(hours=2))
    # If 'gas_price' is fully known for day d at d-1 00:00 => no fill needed
    # but if you do want to fill any missing, you could:
    # forecast_input_data = fill_with_last_week_value(forecast_input_data, test_df, 'gas_price', forecasting_time, known_up_to=pd.Timedelta(hours=24))

    future_data = test_df[(test_df['ds'] >= day_end)]
    # 2.4) Predict
    prediction = nf.predict(df=input_data, futr_df=future_data)

    # Append predictions for evaluation
    predictions.append(prediction)

# Flatten the predictions into a DataFrame
forecast_df = pd.concat(predictions, ignore_index=True)

# Ensure the results align with the test dataset
forecast_df = forecast_df[['ds', model_name]].merge(test_df[['ds', 'y']], on='ds', how='inner')
forecast_df.rename(columns={'y': 'y_actual', model_name: 'y_pred'}, inplace=True)

# Change back to CEST
forecast_df['ds'] = forecast_df['ds'].dt.tz_convert('Europe/Berlin')


In [ ]:
forecast_df.head()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Calculate error metrics
mae = mean_absolute_error(forecast_df['y_actual'], forecast_df['y_pred'])
mse = mean_squared_error(forecast_df['y_actual'], forecast_df['y_pred'])
rmse = np.sqrt(mse)

# Print the results
print("Model Performance:")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

# Merge forecasted values with actual test data, selecting only the relevant columns
comparison_df = forecast_df[['ds', 'y_pred']].merge(test_df[['ds', 'y']], on='ds', how='inner')
comparison_df.rename(columns={'y': 'y_actual', 'y_pred': 'y_pred'}, inplace=True)

# Add a column for the day of the week (0=Monday, 1=Tuesday, ..., 6=Sunday)
comparison_df['day_of_week'] = comparison_df['ds'].dt.dayofweek

# Calculate and display MAE, MSE, and RMSE for each day of the week
metrics_by_day = {}

for day in range(7):
    # Filter data for each day of the week
    day_data = comparison_df[comparison_df['day_of_week'] == day]

    # Calculate the metrics for this day
    mae = mean_absolute_error(day_data['y_actual'], day_data['y_pred'])
    mse = mean_squared_error(day_data['y_actual'], day_data['y_pred'])
    rmse = np.sqrt(mse)

    # Store the metrics for each day
    metrics_by_day[day] = {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse
    }

# Optionally, you can store these results in a DataFrame for easy viewing or further analysis
metrics_df = pd.DataFrame(metrics_by_day).T
print("\nMetrics by Day of the Week:")
print(metrics_df)

Plots

In [ ]:
import matplotlib.pyplot as plt

# Plotting
fig, ax = plt.subplots(1, 1, figsize=(20, 7))

# Plot predictions for model
forecast_df.set_index('ds')['y_pred'].plot(ax=ax, linewidth=2, label= f'{model_name} Prediction')

# Optional: Plot actual test values for comparison
test_df.set_index('ds')['y'].plot(ax=ax, linewidth=2, label='Actual Price')

# Formatting the plot
ax.set_title('Electricity Price Forecast (Day-by-Day)', fontsize=22)
ax.set_ylabel('Price', fontsize=20)
ax.set_xlabel('Date', fontsize=20)
ax.legend(prop={'size': 15})
ax.grid()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Ensure 'ds' is in datetime format
forecast_df['ds'] = pd.to_datetime(forecast_df['ds'])
test_df['ds'] = pd.to_datetime(test_df['ds'])

# Create a new figure with multiple subplots, one for each month
unique_months = forecast_df['ds'].dt.to_period('M').unique()

# Set up the figure and axes for plotting
fig, axes = plt.subplots(len(unique_months), 1, figsize=(20, 7 * len(unique_months)))

if len(unique_months) == 1:
    axes = [axes]  # Ensure axes is iterable even if there's only one subplot

for i, month in enumerate(unique_months):
    # Create the data subset for the current month
    month_str = str(month)
    forecast_month = forecast_df[forecast_df['ds'].dt.to_period('M') == month]
    test_month = test_df[test_df['ds'].dt.to_period('M') == month]

    # Plot predictions for the current month
    axes[i].plot(forecast_month['ds'], forecast_month['y_pred'], linewidth=2, label=f'{model_name} Prediction')

    # Plot actual test values for comparison in the current month
    axes[i].plot(test_month['ds'], test_month['y'], linewidth=2, label='Actual Price')

    # Formatting the plot for the current month
    axes[i].set_title(f'Electricity Price Forecast - {month_str}', fontsize=22)
    axes[i].set_ylabel('Price', fontsize=20)
    axes[i].set_xlabel('Date', fontsize=20)
    axes[i].legend(prop={'size': 15})
    axes[i].grid()

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Ensure 'ds' is in datetime format
forecast_df['ds'] = pd.to_datetime(forecast_df['ds'])
test_df['ds'] = pd.to_datetime(test_df['ds'])

# Create a new figure with multiple subplots, one for each biweekly period
# Group the data by biweekly periods
forecast_df['biweek'] = forecast_df['ds'].dt.to_period('2W')
test_df['biweek'] = test_df['ds'].dt.to_period('2W')

# Get the unique biweekly periods
unique_biweeks = forecast_df['biweek'].unique()

# Set up the figure and axes for plotting
fig, axes = plt.subplots(len(unique_biweeks), 1, figsize=(20, 7 * len(unique_biweeks)))

if len(unique_biweeks) == 1:
    axes = [axes]  # Ensure axes is iterable even if there's only one subplot

for i, biweek in enumerate(unique_biweeks):
    # Create the data subset for the current biweekly period
    biweek_str = str(biweek)
    forecast_biweek = forecast_df[forecast_df['biweek'] == biweek]
    test_biweek = test_df[test_df['biweek'] == biweek]

    # Plot predictions for the current biweekly period
    axes[i].plot(forecast_biweek['ds'], forecast_biweek['y_pred'], linewidth=2, label=f'{model_name} Prediction')

    # Plot actual test values for comparison in the current biweekly period
    axes[i].plot(test_biweek['ds'], test_biweek['y'], linewidth=2, label='Actual Price')

    # Formatting the plot for the current biweekly period
    axes[i].set_title(f'Electricity Price Forecast - {biweek_str}', fontsize=22)
    axes[i].set_ylabel('Price', fontsize=20)
    axes[i].set_xlabel('Date', fontsize=20)
    axes[i].legend(prop={'size': 15})
    axes[i].grid()

# Show the plot
plt.tight_layout()
plt.show()